# 02 - SQL Agent

The goal of this notebook is to build the SQL agent and test it on the two example questions.

The agent combines an LLM, a set of SQL tools and a system prompt. Given a question in plain English, it writes a PostgreSQL query, runs it on the database, and answers in plain English.

For each question, I look at the steps the agent took and check its answer by running my own SQL on the database.

## Setup

The database connection comes from `src/database.py`, the same code as notebook 01, so it's already limited to the two tables.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from src import config
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from src.database import get_database

In [ ]:
db = get_database()

## LLM setup

The agent uses `gpt-4.1` with a temperature of 0, so its queries are as consistent as possible. For SQL, the same question should give the same query.

In [ ]:

llm = ChatOpenAI(model=config.LLM_MODEL,
                       temperature=0)

## SQL toolkit and tools

`SQLDatabaseToolkit` gives the agent four tools for working with the database. The LLM reads each tool's description to decide which one to use and when.

In [ ]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name)
    print(" ", tool.description)
    print()


sql_db_query
  Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema
  Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables
  Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker
  Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



The usual order is: list the tables, get the schema and sample rows for the ones it needs, check the query, then run it. `sql_db_query_checker` is the only tool that uses the LLM, which is why the toolkit needs `llm=llm`.

If a query fails, `sql_db_query` returns the error and its description tells the agent to rewrite the query and try again, so the agent can correct its own mistakes.

`sql_db_query` runs whatever SQL it's given. The "SELECT only" rule is in the system prompt, so it relies on the model following instructions.

## System prompt

The system prompt defines the agent's role, what it may do (read-only SELECT queries), the tables and how they join, the date range of the data, rules for writing queries, and three worked examples. It's stored in `prompts/sql-agent-system-prompt.txt` so it can be edited without changing the code.

In [ ]:
with open(config.PROMPT_FILE, "r", encoding="utf-8") as f:
    system_text = f.read()

In [ ]:
print("Characters:", len(system_text))
print()
print(system_text[:300])


Characters: 4300

ROLE:

You are an expert SQL data analyst, and helpful assistant. Convert the user’s question into an accurate, efficient PostgreSQL 16 SELECT query and a concise natural-language answer.


SCOPE:

- Use read-only SELECT only (never INSERT/UPDATE/DELETE/DDL/admin).

- All tables live in schema groce


## Creating the agent

The agent combines the three parts: the LLM, the four SQL tools, and the system prompt. Given a question, it decides which tools to call, reads the results, and keeps going until it can answer.

In [ ]:
agent = create_agent(model=llm,
                     tools = tools,
                     system_prompt=system_text)

## Question 1: which gender lives furthest from the store?

This question only needs the `customer_details` table. I print the final answer, then every step the agent took.

In [ ]:
user_query = "On average, which gender lives furthest from store?"

result = agent.invoke({"messages": [HumanMessage(content=user_query)]})
print(result["messages"][-1].content)

On average, female customers (gender 'F') live furthest from the store, with an average distance of 3.98 miles.


In [ ]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

On average, which gender lives furthest from store?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_n8kvfhVle4XXpW69pJ657PCX)
 Call ID: call_n8kvfhVle4XXpW69pJ657PCX
  Args:
    query: select
  gender,
  round(avg(distance_from_store), 2) as avg_distance_from_store
from
  grocery_db.customer_details
group by
  gender
order by
  avg_distance_from_store desc;
================================= Tool Message =================================
Name: sql_db_query

[('F', Decimal('3.98')), (None, Decimal('3.36')), ('M', Decimal('0.87'))]
================================== Ai Message ==================================

On average, female customers (gender 'F') live furthest from the store, with an average distance of 3.98 miles.


**Check:** I ran my own query on the database. Females average 3.98 miles (485 customers), customers with no gender 3.36 (5 customers), and males 0.87 (380 customers). The agent's answer is correct.

The agent called `sql_db_query` straight away and skipped the other three tools, because the system prompt already describes both tables. It also skipped the query checker, even though the tool's description says to always use it, so the descriptions guide the agent but don't force it. Its query grouped by gender including NULL, so all three groups were compared before it answered.

In [ ]:
check_query = """
select
  gender,
  round(avg(distance_from_store), 2) as avg_distance,
  count(*) as num_customers
from
  grocery_db.customer_details
group by
  gender
order by
  avg_distance desc;
"""

print(db.run(check_query))

[('F', Decimal('3.98'), 485), (None, Decimal('3.36'), 5), ('M', Decimal('0.87'), 380)]


## Question 2: average transaction value for male customers with a credit score above 0.5, in September 2020

This question needs both tables. It's harder, because each transaction has one row per product area, so a transaction's rows have to be added up before averaging.

In [ ]:
user_query_2 = "What is the average transaction value in September 2020 for male customers who have a credit score above 0.5"

result = agent.invoke({"messages": [HumanMessage(content=user_query_2)]})
print(result["messages"][-1].content)

The average transaction value in September 2020 for male customers with a credit score above 0.5 was $88.49.


In [ ]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the average transaction value in September 2020 for male customers who have a credit score above 0.5
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_rkeoKYhfgZbto08JH7cRcKQg)
 Call ID: call_rkeoKYhfgZbto08JH7cRcKQg
  Args:
    query: with filtered_customers as (
  select customer_id
  from grocery_db.customer_details
  where gender = 'M' and credit_score > 0.5
),
transaction_values as (
  select
    t.transaction_id,
    sum(t.sales_cost) as transaction_value
  from
    grocery_db.transactions t
    join filtered_customers fc on t.customer_id = fc.customer_id
  where
    t.transaction_date >= '2020-09-01'
    and t.transaction_date <= '2020-09-30'
  group by
    t.transaction_id
)
select
  round(avg(transaction_value), 2) as avg_transaction_value
from
  transaction_values;
================================= Tool Message =====

"Average transaction value" can be read in two ways, so I check both:

- **A:** average over all transactions. Each transaction's value is the sum of its rows.
- **B:** each customer's average transaction value first, then the average of those.

In [ ]:
check_a = """
with transaction_values as (
  select
    t.transaction_id,
    sum(t.sales_cost) as transaction_value
  from grocery_db.transactions t
  join grocery_db.customer_details c on t.customer_id = c.customer_id
  where c.gender = 'M'
    and c.credit_score > 0.5
    and t.transaction_date between '2020-09-01' and '2020-09-30'
  group by t.transaction_id
)
select
  round(avg(transaction_value), 2) as avg_transaction_value,
  count(*) as num_transactions
from transaction_values;
"""

check_b = """
with transaction_values as (
  select
    t.customer_id,
    t.transaction_id,
    sum(t.sales_cost) as transaction_value
  from grocery_db.transactions t
  join grocery_db.customer_details c on t.customer_id = c.customer_id
  where c.gender = 'M'
    and c.credit_score > 0.5
    and t.transaction_date between '2020-09-01' and '2020-09-30'
  group by t.customer_id, t.transaction_id
),
avg_per_customer as (
  select customer_id, avg(transaction_value) as avg_value
  from transaction_values
  group by customer_id
)
select
  round(avg(avg_value), 2) as avg_of_customer_averages,
  count(*) as num_customers
from avg_per_customer;
"""

print("A (all transactions):     ", db.run(check_a))
print("B (average per customer): ", db.run(check_b))

A (all transactions):      [(Decimal('88.49'), 1046)]
B (average per customer):  [(Decimal('101.32'), 277)]


The steps output above is long, so here is just the SQL from each tool call, to see which reading the agent chose.

In [ ]:
for message in result["messages"]:
    if message.type == "ai" and message.tool_calls:
        for call in message.tool_calls:
            print(call["name"])
            print(call["args"]["query"])
            print()

sql_db_query_checker
with filtered_customers as (
  select customer_id
  from grocery_db.customer_details
  where gender = 'M' and credit_score > 0.5
),
transaction_values as (
  select
    t.transaction_id,
    sum(t.sales_cost) as transaction_value
  from
    grocery_db.transactions t
    join filtered_customers fc on t.customer_id = fc.customer_id
  where
    t.transaction_date >= '2020-09-01'
    and t.transaction_date <= '2020-09-30'
  group by
    t.transaction_id
)
select
  round(avg(transaction_value), 2) as avg_transaction_value
from
  transaction_values;

sql_db_query
with filtered_customers as (
  select customer_id
  from grocery_db.customer_details
  where gender = 'M' and credit_score > 0.5
),
transaction_values as (
  select
    t.transaction_id,
    sum(t.sales_cost) as transaction_value
  from
    grocery_db.transactions t
    join filtered_customers fc on t.customer_id = fc.customer_id
  where
    t.transaction_date >= '2020-09-01'
    and t.transaction_date <= '2020-

**Check:** I wrote the query two ways, because "average transaction value" can be read in two ways.

| Reading | Result | Based on |
|---|---|---|
| A: average over all transactions (the agent's reading) | $88.49 | 1,046 transactions |
| B: average of each customer's average | $101.32 | 277 customers |

The agent's answer matches A exactly. Its SQL was correct: it filtered male customers with a credit score above 0.5 and September 2020 dates, and summed each transaction's rows before averaging, so nothing was double counted.

But the question was ambiguous, and the agent chose reading A without saying so. Reading B is 14% higher, because customers who shop more often tend to spend less per trip, and A gives them more weight. The system prompt asks the agent to ask a clarifying question when a question is ambiguous, but it didn't treat this one as ambiguous.

This time the agent used `sql_db_query_checker` before running the query, unlike question 1. The checker returned the query unchanged.

## Summary

- The agent answered both questions from the md correctly, which I checked by running my own SQL on the database.
- **Question 1** (which gender lives furthest from the store): females, at 3.98 miles on average. The agent went straight to `sql_db_query` and skipped the other tools, because the system prompt already describes the tables.
- **Question 2** (average transaction value for male customers with credit score above 0.5 in September 2020): $88.49. The agent joined both tables and summed each transaction's rows before averaging, avoiding the double counting that the transactions table makes easy. This time it used the query checker first.
- **The main finding:** question 2 can be read in two ways, which give $88.49 and $101.32. The agent picked one without mentioning the other, even though the system prompt says to ask when a question is ambiguous. Its SQL was correct, but a user wouldn't know another fair answer existed.
- Tool use varies by question: the agent decides when to use each tool, and the tool descriptions guide it but don't force it.

**Next:** move the agent code into `src/agent.py`.